# Tesla Executive Dashboard

Interactive Tesla dashboard for `TSLA`. Run the code cell below. It fetches Yahoo Finance data through `yfinance` when available and falls back to bundled Tesla data offline. The header embeds the current Tesla stylized T logo as an SVG data URI.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from datetime import datetime

# -----------------------------
# Tesla Executive Dashboard
# Live data source: Yahoo Finance via yfinance when available.
# Fallback data is included so the dashboard still runs offline.
# Logo: current Tesla stylized T symbol embedded as SVG data URI.
# -----------------------------
TICKER = "TSLA"
TESLA_LOGO_DATA_URI = "data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+CjwhLS0gR2VuZXJhdG9yOiBBZG9iZSBJbGx1c3RyYXRvciAxNS4wLjAsIFNWRyBFeHBvcnQgUGx1Zy1JbiAuIFNWRyBWZXJzaW9uOiA2LjAwIEJ1aWxkIDApICAtLT4KCjxzdmcKICAgeG1sbnM6ZGM9Imh0dHA6Ly9wdXJsLm9yZy9kYy9lbGVtZW50cy8xLjEvIgogICB4bWxuczpjYz0iaHR0cDovL2NyZWF0aXZlY29tbW9ucy5vcmcvbnMjIgogICB4bWxuczpyZGY9Imh0dHA6Ly93d3cudzMub3JnLzE5OTkvMDIvMjItcmRmLXN5bnRheC1ucyMiCiAgIHhtbG5zOnN2Zz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciCiAgIHhtbG5zPSJodHRwOi8vd3d3LnczLm9yZy8yMDAwL3N2ZyIKICAgeG1sbnM6c29kaXBvZGk9Imh0dHA6Ly9zb2RpcG9kaS5zb3VyY2Vmb3JnZS5uZXQvRFREL3NvZGlwb2RpLTAuZHRkIgogICB4bWxuczppbmtzY2FwZT0iaHR0cDovL3d3dy5pbmtzY2FwZS5vcmcvbmFtZXNwYWNlcy9pbmtzY2FwZSIKICAgdmVyc2lvbj0iMS4xIgogICBpZD0iTGF5ZXJfMSIKICAgeD0iMHB4IgogICB5PSIwcHgiCiAgIHdpZHRoPSIyNTQuNTg0IgogICBoZWlnaHQ9IjI1My41MDIiCiAgIHZpZXdCb3g9IjAgMCAyNTQuNTg0MDEgMjUzLjUwMiIKICAgZW5hYmxlLWJhY2tncm91bmQ9Im5ldyAwIDAgMzQ1Ljg2IDUwMCIKICAgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIKICAgaW5rc2NhcGU6dmVyc2lvbj0iMC45Mi4wIHIxNTI5OSIKICAgc29kaXBvZGk6ZG9jbmFtZT0iVGVzbGFfTW90b3JzLnN2ZyI+PG1ldGFkYXRhCiAgICAgaWQ9Im1ldGFkYXRhNDMiPjxyZGY6UkRGPjxjYzpXb3JrCiAgICAgICAgIHJkZjphYm91dD0iIj48ZGM6Zm9ybWF0PmltYWdlL3N2Zyt4bWw8L2RjOmZvcm1hdD48ZGM6dHlwZQogICAgICAgICAgIHJkZjpyZXNvdXJjZT0iaHR0cDovL3B1cmwub3JnL2RjL2RjbWl0eXBlL1N0aWxsSW1hZ2UiIC8+PGRjOnRpdGxlPjwvZGM6dGl0bGU+PC9jYzpXb3JrPjwvcmRmOlJERj48L21ldGFkYXRhPjxkZWZzCiAgICAgaWQ9ImRlZnM0MSIgLz48c29kaXBvZGk6bmFtZWR2aWV3CiAgICAgcGFnZWNvbG9yPSIjZmZmZmZmIgogICAgIGJvcmRlcmNvbG9yPSIjNjY2NjY2IgogICAgIGJvcmRlcm9wYWNpdHk9IjEiCiAgICAgb2JqZWN0dG9sZXJhbmNlPSIxMCIKICAgICBncmlkdG9sZXJhbmNlPSIxMCIKICAgICBndWlkZXRvbGVyYW5jZT0iMTAiCiAgICAgaW5rc2NhcGU6cGFnZW9wYWNpdHk9IjAiCiAgICAgaW5rc2NhcGU6cGFnZXNoYWRvdz0iMiIKICAgICBpbmtzY2FwZTp3aW5kb3ctd2lkdGg9IjE5MjAiCiAgICAgaW5rc2NhcGU6d2luZG93LWhlaWdodD0iMTA1NyIKICAgICBpZD0ibmFtZWR2aWV3MzkiCiAgICAgc2hvd2dyaWQ9ImZhbHNlIgogICAgIGlua3NjYXBlOnNob3dwYWdlc2hhZG93PSJmYWxzZSIKICAgICBib3JkZXJsYXllcj0iZmFsc2UiCiAgICAgZml0LW1hcmdpbi10b3A9IjAuNSIKICAgICBmaXQtbWFyZ2luLWxlZnQ9IjAuNSIKICAgICBmaXQtbWFyZ2luLXJpZ2h0PSIwLjUiCiAgICAgZml0LW1hcmdpbi1ib3R0b209IjAuNSIKICAgICBpbmtzY2FwZTp6b29tPSIxLjA0MjM0MDciCiAgICAgaW5rc2NhcGU6Y3g9Ii0xODkuOTk3NyIKICAgICBpbmtzY2FwZTpjeT0iMTA3Ljc3OTM1IgogICAgIGlua3NjYXBlOndpbmRvdy14PSItOCIKICAgICBpbmtzY2FwZTp3aW5kb3cteT0iLTgiCiAgICAgaW5rc2NhcGU6d2luZG93LW1heGltaXplZD0iMSIKICAgICBpbmtzY2FwZTpjdXJyZW50LWxheWVyPSJUIiAvPjxnCiAgICAgaWQ9IlQiCiAgICAgdHJhbnNmb3JtPSJ0cmFuc2xhdGUoLTQ1Ljg0LC02NC4yOTcpIgogICAgIHN0eWxlPSJzdHJva2U6bm9uZSI+PHBhdGgKICAgICAgIGQ9Ik0gMTczLjE0NiwzMTcuMjk5IDIwOC42MjIsMTE3Ljc4IGMgMzMuODE1LDAgNDQuNDgxLDMuNzA4IDQ2LjAyMSwxOC44NDMgMCwwIDIyLjY4NCwtOC40NTggMzQuMTI1LC0yNS42MzYgQyAyNDQuMTIyLDkwLjI5OSAxOTkuMjYzLDg5LjM2NiAxOTkuMjYzLDg5LjM2NiBsIC0yNi4xNzYsMzEuODgyIDAuMDU5LC0wLjAwNCAtMjYuMTc2LC0zMS44ODMgYyAwLDAgLTQ0Ljg2LDAuOTM0IC04OS41LDIxLjYyMiAxMS40MzEsMTcuMTc4IDM0LjEyNCwyNS42MzYgMzQuMTI0LDI1LjYzNiAxLjU0OSwtMTUuMTM2IDEyLjIwMiwtMTguODQ0IDQ1Ljc5LC0xOC44NjggbCAzNS43NjIsMTk5LjU0OCIKICAgICAgIGlkPSJwYXRoMzUiCiAgICAgICBzdHlsZT0iZmlsbDojZTgyMTI3O2ZpbGwtb3BhY2l0eToxO3N0cm9rZTpub25lIgogICAgICAgaW5rc2NhcGU6Y29ubmVjdG9yLWN1cnZhdHVyZT0iMCIgLz48cGF0aAogICAgICAgZD0ibSAxNzMuMTMyLDgwLjE1NyBjIDM2LjA5LC0wLjI3NiA3Ny4zOTksNS41ODMgMTE5LjY4NywyNC4wMTQgNS42NTIsLTEwLjE3MyA3LjEwNSwtMTQuNjY5IDcuMTA1LC0xNC42NjkgQyAyNTMuNjk3LDcxLjIxMyAyMTAuNDA2LDY0Ljk1NCAxNzMuMTI3LDY0Ljc5NyAxMzUuODUsNjQuOTU0IDkyLjU2MSw3MS4yMTQgNDYuMzQsODkuNTAyIGMgMCwwIDIuMDYyLDUuNTM4IDcuMSwxNC42NjkgNDIuMjgsLTE4LjQzMSA4My41OTYsLTI0LjI5IDExOS42ODcsLTI0LjAxNCBoIDAuMDA1IgogICAgICAgaWQ9InBhdGgzNyIKICAgICAgIHN0eWxlPSJmaWxsOiNlODIxMjc7ZmlsbC1vcGFjaXR5OjE7c3Ryb2tlOm5vbmUiCiAgICAgICBpbmtzY2FwZTpjb25uZWN0b3ItY3VydmF0dXJlPSIwIiAvPjwvZz48L3N2Zz4="

fallback_df = pd.DataFrame({
    "Year": [2019, 2020, 2021, 2022, 2023, 2024],
    "Revenue": [24578, 31536, 53823, 81462, 96773, 97690],
    "Net Income": [-862, 721, 5519, 12583, 14974, 7130],
    "Gross Profit": [4069, 6630, 13606, 20853, 17660, 17317],
    "Automotive": [20821, 27236, 47232, 71462, 82419, 77070],
    "Energy": [1531, 1994, 2789, 3909, 6035, 10086],
    "Services": [2226, 2306, 3802, 6091, 8319, 10534],
    "Cash": [6268, 19384, 17576, 22185, 29094, 36563],
    "Debt": [13419, 11739, 8873, 5748, 9573, 12628],
    "Market Cap": [76000, 668000, 1061000, 389000, 789000, 1296000],
})

fallback_stock = pd.DataFrame({
    "Date": pd.to_datetime(["2019-12-31", "2020-12-31", "2021-12-31", "2022-12-31", "2023-12-31", "2024-12-31"]),
    "Close": [28, 235, 352, 123, 248, 404],
})

DATA_SOURCE = "Offline fallback Tesla data"
LAST_UPDATED = datetime.now().strftime("%Y-%m-%d %H:%M")

def _to_millions(value):
    if pd.isna(value):
        return np.nan
    return float(value) / 1_000_000

def _row_value(frame, row_names, column):
    for row_name in row_names:
        if row_name in frame.index:
            value = frame.loc[row_name, column]
            if isinstance(value, pd.Series):
                value = value.iloc[0]
            return _to_millions(value)
    return np.nan

def load_yahoo_tesla_data():
    """Fetch annual Tesla financials and stock prices from Yahoo Finance if yfinance is installed."""
    try:
        import yfinance as yf
    except Exception as exc:
        raise RuntimeError("Install yfinance to fetch live Yahoo Finance data: pip install yfinance") from exc

    ticker = yf.Ticker(TICKER)
    financials = ticker.financials
    balance_sheet = ticker.balance_sheet

    if financials is None or financials.empty:
        raise RuntimeError("Yahoo Finance did not return annual financial statements.")

    rows = []
    for col in sorted(financials.columns):
        year = int(pd.Timestamp(col).year)
        revenue = _row_value(financials, ["Total Revenue"], col)
        net_income = _row_value(financials, ["Net Income", "Net Income Common Stockholders"], col)
        gross_profit = _row_value(financials, ["Gross Profit"], col)

        cash = np.nan
        debt = np.nan
        if balance_sheet is not None and not balance_sheet.empty and col in balance_sheet.columns:
            cash = _row_value(balance_sheet, ["Cash And Cash Equivalents", "Cash Cash Equivalents And Short Term Investments"], col)
            debt = _row_value(balance_sheet, ["Total Debt", "Long Term Debt And Capital Lease Obligation"], col)

        rows.append({
            "Year": year,
            "Revenue": revenue,
            "Net Income": net_income,
            "Gross Profit": gross_profit,
            "Cash": cash,
            "Debt": debt,
        })

    yahoo_df = pd.DataFrame(rows).dropna(subset=["Revenue"]).sort_values("Year").tail(7)
    if yahoo_df.empty:
        raise RuntimeError("Yahoo Finance returned no usable annual rows.")

    # Yahoo Finance statements do not reliably expose Tesla segment revenue by product line,
    # so segment rows use Tesla-style reported categories from the fallback set where available.
    segment_cols = ["Automotive", "Energy", "Services"]
    yahoo_df = yahoo_df.merge(fallback_df[["Year"] + segment_cols], on="Year", how="left")
    for col in segment_cols:
        if yahoo_df[col].isna().any():
            ratio = fallback_df[col].sum() / fallback_df["Revenue"].sum()
            yahoo_df[col] = yahoo_df[col].fillna(yahoo_df["Revenue"] * ratio)

    try:
        fast_info = ticker.fast_info
        market_cap = getattr(fast_info, "market_cap", None) or fast_info.get("market_cap")
    except Exception:
        market_cap = None
    yahoo_df["Market Cap"] = np.nan
    yahoo_df.loc[yahoo_df.index[-1], "Market Cap"] = _to_millions(market_cap) if market_cap else fallback_df["Market Cap"].iloc[-1]
    yahoo_df["Market Cap"] = yahoo_df["Market Cap"].interpolate().bfill().ffill()

    stock = ticker.history(period="5y", interval="1mo", auto_adjust=True).reset_index()
    if stock is None or stock.empty:
        stock = fallback_stock.copy()
    stock["Date"] = pd.to_datetime(stock["Date"]).dt.tz_localize(None)

    return yahoo_df.reset_index(drop=True), stock[["Date", "Close"]].dropna(), "Yahoo Finance via yfinance"

try:
    df, stock_df, DATA_SOURCE = load_yahoo_tesla_data()
except Exception as err:
    df = fallback_df.copy()
    stock_df = fallback_stock.copy()
    DATA_SOURCE = f"Offline fallback Tesla data ({err})"

metric_cols = ["Revenue", "Net Income", "Gross Profit", "Automotive", "Energy", "Services", "Cash", "Debt", "Market Cap"]
for col in metric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

def pct_change(data, column, selected_value):
    if selected_value == "All":
        first_value = data.loc[data["Year"] == data["Year"].min(), column].iloc[0]
        last_value = data.loc[data["Year"] == data["Year"].max(), column].iloc[0]
        if first_value == 0:
            return "No base value"
        change = (last_value - first_value) / abs(first_value) * 100
        return f"{change:+.1f}% overall change"

    current = data.loc[data["Year"] == selected_value, column].iloc[0]
    previous = data.loc[data["Year"] < selected_value, column]
    if previous.empty or previous.iloc[-1] == 0:
        return "Base year"
    change = (current - previous.iloc[-1]) / abs(previous.iloc[-1]) * 100
    return f"{change:+.1f}% vs previous year"

def money(value):
    return f"$ {value:,.0f} M"

def draw_dashboard(selected_year):
    if selected_year == "All":
        row = df.iloc[-1]
        data = df.copy()
        dashboard_title = "All Years Combined"
    else:
        row = df[df["Year"] == selected_year].iloc[0]
        data = df[df["Year"] <= selected_year]
        dashboard_title = selected_year

    display(HTML(f"""
    <div style="
        background:linear-gradient(135deg,#050505,#111 48%,#2A0507);
        border:1px solid #2F2F2F;
        border-radius:8px;
        padding:20px 24px;
        font-family:Arial, Helvetica, sans-serif;
        color:white;
        margin-bottom:14px;
        box-shadow:0 14px 30px rgba(0,0,0,.34);
    ">
        <div style="display:flex; align-items:center; gap:24px; flex-wrap:wrap;">
            <div style="
                background:#FFFFFF;
                color:#050505;
                padding:16px 22px;
                border-radius:8px;
                min-width:270px;
                box-shadow:0 8px 24px rgba(232,33,39,.22);
            ">
                <div style="display:flex; align-items:center; justify-content:center; gap:12px;">
                    <img src="{TESLA_LOGO_DATA_URI}" alt="Tesla stylized T logo" style="width:58px; height:58px; object-fit:contain;" />
                    <div style="font-size:36px; font-weight:900; letter-spacing:4px;">TESLA</div>
                </div>
                <div style="text-align:center; font-size:13px; color:#333; font-weight:800; margin-top:4px;">
                    Electric Vehicles, Energy and AI
                </div>
            </div>
            <div style="flex:1; min-width:320px;">
                <div style="font-size:32px; font-weight:900; letter-spacing:0;">Tesla Executive Dashboard</div>
                <div style="font-size:15px; color:#D1D5DB; margin-top:6px;">
                    Revenue, profitability, segment mix, energy growth, liquidity, market value and TSLA stock movement
                </div>
                <div style="font-size:12px; color:#E82127; margin-top:8px; font-weight:800;">
                    Source: {DATA_SOURCE} | Refreshed: {LAST_UPDATED}
                </div>
            </div>
        </div>
    </div>
    """))

    if selected_year == "All":
        revenue_value = df["Revenue"].sum()
        income_value = df["Net Income"].sum()
        energy_value = df["Energy"].sum()
        market_value = df["Market Cap"].iloc[-1]
    else:
        revenue_value = row["Revenue"]
        income_value = row["Net Income"]
        energy_value = row["Energy"]
        market_value = row["Market Cap"]

    kpis = [
        ("Revenue", money(revenue_value), pct_change(df, "Revenue", selected_year), "#FFFFFF"),
        ("Net Income", money(income_value), pct_change(df, "Net Income", selected_year), "#E82127"),
        ("Energy", money(energy_value), pct_change(df, "Energy", selected_year), "#22C55E"),
        ("Market Cap", money(market_value), pct_change(df, "Market Cap", selected_year), "#FCA5A5"),
    ]

    kpi_html = ""
    for title, value, delta, color in kpis:
        kpi_html += f"""
        <div style="
            background:linear-gradient(145deg,#0A0A0A,#171717);
            border:1px solid #2D2D2D;
            border-radius:8px;
            padding:16px;
            color:white;
            box-shadow:0 10px 24px rgba(0,0,0,.22);
        ">
            <div style="font-size:12px; color:#9CA3AF; font-weight:900; text-transform:uppercase;">{title}</div>
            <div style="font-size:28px; font-weight:900; margin-top:8px; color:{color};">{value}</div>
            <div style="font-size:13px; color:#D1D5DB; margin-top:6px;">{delta}</div>
        </div>
        """

    display(HTML(f"""
    <div style="display:grid; grid-template-columns:repeat(4,minmax(160px,1fr)); gap:14px; font-family:Arial, Helvetica, sans-serif; margin-bottom:16px;">
        {kpi_html}
    </div>
    """))

    plt.style.use("dark_background")
    fig = plt.figure(figsize=(18, 10), facecolor="#050505")
    gs = fig.add_gridspec(2, 3, hspace=0.38, wspace=0.28)

    chart_bg = "#0A0A0A"
    grid_color = "#303030"
    text_color = "#E5E7EB"
    white = "#FFFFFF"
    tesla_red = "#E82127"
    red_soft = "#FCA5A5"
    green = "#22C55E"
    blue = "#38BDF8"
    amber = "#FBBF24"
    gray = "#9CA3AF"

    def style_ax(ax, title):
        ax.set_facecolor(chart_bg)
        ax.set_title(title, color="white", fontsize=14, fontweight="bold", pad=12)
        ax.tick_params(colors=text_color)
        ax.grid(True, color=grid_color, alpha=0.55)
        for spine in ax.spines.values():
            spine.set_color("#3A3A3A")

    ax1 = fig.add_subplot(gs[0, 0])
    style_ax(ax1, "Revenue and Net Income Trend")
    ax1.bar(data["Year"], data["Revenue"], color=tesla_red, label="Revenue")
    ax1.set_ylabel("Revenue $M", color=tesla_red)
    ax1.tick_params(axis="y", labelcolor=tesla_red)

    ax1_income = ax1.twinx()
    ax1_income.plot(data["Year"], data["Net Income"], color=white, marker="o", linewidth=3, label="Net Income")
    ax1_income.axhline(0, color=gray, linewidth=1, alpha=0.7)
    ax1_income.set_ylabel("Net Income $M", color=white)
    ax1_income.tick_params(axis="y", labelcolor=white)
    ax1_income.set_facecolor(chart_bg)
    for spine in ax1_income.spines.values():
        spine.set_color("#3A3A3A")
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax1_income.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, facecolor=chart_bg, edgecolor="#3A3A3A", loc="upper left")

    ax2 = fig.add_subplot(gs[0, 1])
    ax2.set_facecolor(chart_bg)
    ax2.set_title("Business Segment Mix" if selected_year == "All" else f"Business Segment Mix - {selected_year}", color="white", fontsize=14, fontweight="bold", pad=12)
    segment_cols = ["Automotive", "Energy", "Services"]
    segment_values = [df[col].sum() if selected_year == "All" else row[col] for col in segment_cols]
    wedges, texts, autotexts = ax2.pie(
        segment_values,
        labels=["Automotive", "Energy", "Services"],
        autopct="%1.1f%%",
        startangle=90,
        colors=[tesla_red, green, blue],
        wedgeprops=dict(width=0.42, edgecolor=chart_bg),
    )
    for t in texts + autotexts:
        t.set_color("white")
        t.set_fontsize(9)
    ax2.text(0, 0, "TSLA", ha="center", va="center", color="white", fontsize=16, fontweight="bold")

    ax3 = fig.add_subplot(gs[0, 2])
    style_ax(ax3, "Segment Revenue Trend")
    ax3.plot(data["Year"], data["Automotive"], color=tesla_red, marker="o", linewidth=3, label="Automotive")
    ax3.plot(data["Year"], data["Energy"], color=green, marker="o", linewidth=3, label="Energy")
    ax3.plot(data["Year"], data["Services"], color=blue, marker="o", linewidth=3, label="Services")
    ax3.set_ylabel("$M")
    ax3.legend(facecolor=chart_bg, edgecolor="#3A3A3A", fontsize=8)

    ax4 = fig.add_subplot(gs[1, 0])
    style_ax(ax4, "Gross Profit, Cash and Debt")
    ax4.plot(data["Year"], data["Gross Profit"], color=amber, marker="o", linewidth=3, label="Gross Profit")
    ax4.fill_between(data["Year"], data["Cash"], color=green, alpha=0.35, label="Cash")
    ax4.fill_between(data["Year"], data["Debt"], color=red_soft, alpha=0.32, label="Debt")
    ax4.set_ylabel("$M")
    ax4.legend(facecolor=chart_bg, edgecolor="#3A3A3A", fontsize=8)

    ax5 = fig.add_subplot(gs[1, 1])
    style_ax(ax5, "Margin Analysis")
    gross_margin = data["Gross Profit"] / data["Revenue"].replace(0, np.nan) * 100
    net_margin = data["Net Income"] / data["Revenue"].replace(0, np.nan) * 100
    ax5.plot(data["Year"], gross_margin, color=tesla_red, marker="o", linewidth=3, label="Gross Margin")
    ax5.plot(data["Year"], net_margin, color=white, marker="o", linewidth=3, label="Net Margin")
    ax5.axhline(0, color=gray, linewidth=1, alpha=0.7)
    ax5.set_ylabel("%")
    ax5.legend(facecolor=chart_bg, edgecolor="#3A3A3A")

    ax6 = fig.add_subplot(gs[1, 2])
    style_ax(ax6, "TSLA Stock Price - 5Y Monthly")
    filtered_stock = stock_df.copy()
    if selected_year != "All":
        filtered_stock = filtered_stock[filtered_stock["Date"].dt.year <= selected_year]
    ax6.plot(filtered_stock["Date"], filtered_stock["Close"], color=tesla_red, linewidth=3, label="Adjusted Close")
    ax6.fill_between(filtered_stock["Date"], filtered_stock["Close"], color=tesla_red, alpha=0.18)
    ax6.set_ylabel("Price $")
    ax6.legend(facecolor=chart_bg, edgecolor="#3A3A3A", fontsize=8)

    fig.suptitle(f"Tesla Dashboard - {dashboard_title}", fontsize=22, fontweight="bold", color="white", y=0.99)
    plt.show()

year_filter = widgets.Dropdown(
    options=["All"] + sorted(df["Year"].astype(int).tolist(), reverse=True),
    value="All",
    description="Year Filter:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="260px")
)

dashboard_output = widgets.Output()

def update_dashboard(change=None):
    with dashboard_output:
        clear_output(wait=True)
        draw_dashboard(year_filter.value)

year_filter.observe(update_dashboard, names="value")

display(year_filter)
display(dashboard_output)
update_dashboard()
